# Arabic Ontology Chunker — Demo

**عربي:** هذا الدفتر يوضح نظام التقطيع الأنطولوجي للنصوص العربية — يعمل بالكامل دون إنترنت أو مفاتيح API.

**English:** This notebook walks through the full offline Arabic chunking pipeline and compares it against three baselines.

---
**Sections**
1. Setup & imports
2. Basic usage — chunk a news paragraph
3. Islamic text — tafsir and concept boundaries
4. Side-by-side comparison of all 4 methods
5. RAG demo — retrieve answers from QA pairs
6. Results visualisation — bar chart

---
## 1 — Setup & Imports

**عربي:** إعداد المسارات واستيراد المكتبات.

In [ ]:
import sys, os
from pathlib import Path

# Make the repo root importable
REPO = Path(os.getcwd()).parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.chunker.ontology_chunker import OntologyChunker
from src.chunker.concept_tagger import ConceptTagger
from src.baselines.fixed_chunker import FixedChunker
from src.baselines.recursive_chunker import RecursiveChunker
from src.preprocessing.normalizer import normalize
from src.preprocessing.tokenizer import sentence_tokenize

import json

DATA = REPO / 'data'

print('Imports OK')
tagger = ConceptTagger(use_morphology=False)
print('Loaded domains:', tagger.domain_names)

---
## 2 — Basic Usage: Chunk a News Paragraph

**عربي:** نُقطّع مقالاً إخبارياً يتناول السياسة ثم الاقتصاد ونلاحظ أين يضع النظام حدوداً.

**English:** Chunk a news article that mixes politics and economics, and inspect where boundaries are placed.

In [ ]:
article_1 = (DATA / 'news' / 'article_1.txt').read_text(encoding='utf-8')

chunker = OntologyChunker(shift_threshold=0.5, min_sentences=2, normalize_input=True)
chunks = chunker.chunk_rich(article_1)

print(f'Total chunks: {len(chunks)}\n')
for c in chunks:
    print(f"━━ Chunk {c.chunk_index}  [{c.concept} / {c.concept_en}]  "
          f"confidence={c.confidence:.2f}  sentences={c.sentence_count}")
    print(f"   Boundary: {c.boundary_reason}")
    print(f"   Keywords: {c.keywords[:5]}")
    preview = c.text[:150].replace('\n', ' ')
    print(f"   Text: {preview}…\n")

**Expected output:** Two chunks — one labelled `سياسة / Politics`, one `اقتصاد / Economics` — matching the natural topic transition in the article.

---
## 3 — Islamic Text: Tafsir and Concept Boundaries

**عربي:** نختبر النظام على نص تفسيري يمزج بين أحكام الصلاة وأحكام الصيام.

**English:** The tafsir sample covers prayer (الصلاة) then fasting (الصيام).  The chunker should detect the topic shift and place a boundary between them.

In [ ]:
tafsir = (DATA / 'islamic' / 'quran_tafsir_sample.txt').read_text(encoding='utf-8')

# Use a specialised chunker tuned for religious text
islamic_chunker = OntologyChunker(
    shift_threshold=0.45,   # slightly lower — religious concepts overlap
    min_sentences=2,
    max_sentences=12,
)
chunks = islamic_chunker.chunk_rich(tafsir)

print(f'Tafsir chunks: {len(chunks)}\n')
for c in chunks:
    print(f"━━ Chunk {c.chunk_index}  [{c.concept}]  conf={c.confidence:.2f}")
    print(f"   Reason : {c.boundary_reason}")
    print(f"   سنتين مثال: {' '.join(c.sentences[:2])}\n")

In [ ]:
# Inspect a single sliding window manually
sentences = sentence_tokenize(normalize(tafsir))
print(f'Total sentences: {len(sentences)}\n')

for i in range(0, min(len(sentences) - 2, 6)):
    window = sentences[i:i+3]
    result = tagger.tag(window)
    print(f'Window [{i}–{i+2}]  →  {result.concept} ({result.concept_en})  '
          f'conf={result.confidence:.2f}  kw={result.keywords[:3]}')

---
## 4 — Side-by-Side Comparison of All 4 Methods

**عربي:** نُطبّق الطرق الأربع على النص نفسه ونقارن النتائج في جدول.

**English:** Apply all four chunkers to article_2 (sports → technology) and display results in a table.

In [ ]:
article_2 = (DATA / 'news' / 'article_2.txt').read_text(encoding='utf-8')

methods = {
    'OntologyChunker':  OntologyChunker(shift_threshold=0.5, min_sentences=2),
    'FixedChunker':     FixedChunker(chunk_size=100, overlap=10),
    'RecursiveChunker': RecursiveChunker(chunk_size=300, overlap=30),
}

comparison = {}
for name, m in methods.items():
    dicts = m.chunk_dicts(article_2)
    comparison[name] = dicts

# Summary table
print(f'{"Method":<22} {"# Chunks":>9} {"Avg Words":>10} {"Concept-Aware":>14}')
print('-' * 58)
for name, chunks in comparison.items():
    avg_words = sum(len(c['text'].split()) for c in chunks) / max(len(chunks), 1)
    concept_aware = 'Yes' if chunks[0]['concept'] is not None else 'No'
    print(f'{name:<22} {len(chunks):>9} {avg_words:>10.1f} {concept_aware:>14}')

In [ ]:
# Show first chunk from each method
print('=== First chunk from each method ===\n')
for name, chunks in comparison.items():
    c = chunks[0]
    preview = c['text'][:200].replace('\n', ' ')
    concept_label = f"[{c['concept']} / {c['concept_en']}]" if c['concept'] else '[no concept]'
    print(f'── {name}  {concept_label}')
    print(f'   {preview}…')
    print(f'   ({c["sentence_count"]} sentences | {c["boundary_reason"]})\n')

---
## 5 — RAG Demo: Retrieve Answers Using OntologyChunker

**عربي:** نُحمّل أزواج الأسئلة والأجوبة ثم نحاول استرجاع الإجابة من المقاطع المُقطَّعة باستخدام بحث نصي بسيط.

**English:** Load QA pairs, chunk the source documents, and check whether the answer is found in the top retrieved chunks (using simple keyword search as a proxy for embedding-based retrieval).

In [ ]:
qa_pairs = json.loads((DATA / 'qa_pairs.json').read_text(encoding='utf-8'))

# Build a chunk store: {source_file: [chunk_dicts]}
chunker = OntologyChunker(shift_threshold=0.5, min_sentences=2)
chunk_store = {}
for fname in ['news/article_1.txt', 'news/article_2.txt', 'news/article_3.txt',
              'islamic/quran_tafsir_sample.txt', 'islamic/hadith_sample.txt']:
    fpath = DATA / Path(fname)
    if fpath.exists():
        text = fpath.read_text(encoding='utf-8')
        chunk_store[fname] = chunker.chunk_dicts(text)

print(f'Files indexed: {len(chunk_store)}')
for k, v in chunk_store.items():
    print(f'  {k}: {len(v)} chunks')

In [ ]:
def keyword_retrieve(chunks, query, k=3):
    """Simple keyword overlap retrieval (proxy for embedding search)."""
    query_words = set(query.split())
    scored = []
    for c in chunks:
        chunk_words = set(c['text'].split())
        score = len(query_words & chunk_words) / max(len(query_words), 1)
        scored.append((score, c))
    scored.sort(key=lambda x: -x[0])
    return [c for _, c in scored[:k]]

# Evaluate on all QA pairs
hits_at_1 = []
hits_at_3 = []

for qa in qa_pairs:
    source = qa['source_file']
    chunks = chunk_store.get(source, [])
    retrieved = keyword_retrieve(chunks, qa['question'], k=3)
    
    answer = qa['answer'].lower()
    found_at_1 = any(answer in c['text'].lower() for c in retrieved[:1])
    found_at_3 = any(answer in c['text'].lower() for c in retrieved[:3])
    hits_at_1.append(int(found_at_1))
    hits_at_3.append(int(found_at_3))

p1 = sum(hits_at_1) / len(hits_at_1)
p3 = sum(hits_at_3) / len(hits_at_3)
print(f'OntologyChunker + Keyword Search')
print(f'  Precision@1 = {p1:.3f}  ({sum(hits_at_1)}/{len(hits_at_1)})')
print(f'  Precision@3 = {p3:.3f}  ({sum(hits_at_3)}/{len(hits_at_3)})')

In [ ]:
# Show sample retrievals
print('=== Sample QA Retrievals ===\n')
for qa in qa_pairs[:4]:
    source = qa['source_file']
    chunks = chunk_store.get(source, [])
    retrieved = keyword_retrieve(chunks, qa['question'], k=3)
    found = any(qa['answer'].lower() in c['text'].lower() for c in retrieved[:3])
    
    print(f"Q: {qa['question']}")
    print(f"A: {qa['answer']}")
    print(f"Found in top-3: {'✓' if found else '✗'}")
    if retrieved:
        top = retrieved[0]
        concept_label = f"[{top['concept']}]" if top.get('concept') else ''
        print(f"Top chunk {concept_label}: {top['text'][:100]}…")
    print()

---
## 6 — Results Visualisation

**عربي:** رسم بياني يقارن أداء الطرق الأربع.

**English:** Bar chart comparing Precision@3 and concept purity across all four chunkers on the sample documents.

In [ ]:
import math
from src.evaluation.metrics import chunk_coherence_score, concept_purity

# Compare quality metrics on article_1 across methods
article_1 = (DATA / 'news' / 'article_1.txt').read_text(encoding='utf-8')
tagger = ConceptTagger(use_morphology=False)

quality = {}
for name, m in methods.items():
    dicts = m.chunk_dicts(article_1)
    coh = chunk_coherence_score(dicts, tagger)
    pur = concept_purity(dicts, tagger)
    quality[name] = {'coherence': coh, 'purity': pur}
    print(f'{name:<22}  coherence={coh:.3f}  purity={pur:.3f}')

In [ ]:
try:
    import matplotlib
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.rcParams['font.family'] = 'DejaVu Sans'

    names     = list(quality.keys())
    coherence = [quality[n]['coherence'] if not math.isnan(quality[n]['coherence']) else 0 for n in names]
    purity    = [quality[n]['purity']    if not math.isnan(quality[n]['purity'])    else 0 for n in names]

    x = range(len(names))
    width = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Coherence
    axes[0].bar(x, coherence, width, color=['#2ecc71', '#3498db', '#e74c3c'], alpha=0.85)
    axes[0].set_xticks(list(x))
    axes[0].set_xticklabels(names, rotation=15, ha='right')
    axes[0].set_ylim(0, 1)
    axes[0].set_ylabel('Avg Concept Confidence')
    axes[0].set_title('Chunk Coherence Score')
    axes[0].grid(axis='y', alpha=0.3)

    # Purity
    axes[1].bar(x, purity, width, color=['#2ecc71', '#3498db', '#e74c3c'], alpha=0.85)
    axes[1].set_xticks(list(x))
    axes[1].set_xticklabels(names, rotation=15, ha='right')
    axes[1].set_ylim(0, 1)
    axes[1].set_ylabel('Fraction of Same-Concept Sentence Pairs')
    axes[1].set_title('Concept Purity')
    axes[1].grid(axis='y', alpha=0.3)

    plt.suptitle('Arabic Chunker Quality Metrics (Article 1: Politics → Economics)', fontsize=13)
    plt.tight_layout()
    plt.savefig(REPO / 'notebooks' / 'benchmark_chart.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Chart saved to notebooks/benchmark_chart.png')

except ImportError:
    print('matplotlib not installed — skipping chart.  pip install matplotlib')

---
## Adding a New Domain

**عربي:** لإضافة مجال جديد، أنشئ ملف JSON في `data/domains/` دون أي تعديل على الكود.

**English:** Drop a JSON file into `data/domains/` — no code changes needed.

In [ ]:
import json, tempfile
from src.chunker.concept_tagger import ConceptTagger

sports_domain = {
    "concept":    "رياضة",
    "concept_en": "Sports",
    "keywords":   ["كرة", "ملعب", "لاعب", "بطولة", "هدف", "مباراة", "فريق", "مدرب"],
    "roots":      ["كرر", "لعب", "طول", "هدف", "برو", "درب"]
}

# Write to a temp file (in practice: put it in data/domains/sports.json)
with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False, encoding='utf-8') as tf:
    json.dump(sports_domain, tf, ensure_ascii=False)
    tmp_path = tf.name

tagger_with_sports = ConceptTagger(use_morphology=False)
tagger_with_sports.add_domain(tmp_path)
print('Domains after adding Sports:', tagger_with_sports.domain_names)

sports_window = ['سجّل المهاجم هدفاً رائعاً في المباراة.', 'فاز الفريق بثلاثة أهداف.', 'غادر المدرب الملعب سعيداً.']
result = tagger_with_sports.tag(sports_window)
print(f'\nSports window → concept: {result.concept} ({result.concept_en}), confidence: {result.confidence}')